# AI/ML-Based Intrusion Detection System (IDS)
## Exploratory Data Analysis & Model Comparison

This notebook provides a comprehensive analysis of network traffic data and compares machine learning models for intrusion detection.

**Author**: blackcop1  
**Date**: 2026  
**Project**: AI/ML-Based IDS for Network Anomaly Detection

## 1. Setup & Imports

In [ ]:
# Standard libraries
import sys
import os
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd()))

# Data processing
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import graph_objects as go
import plotly.express as px

# Machine Learning
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Project imports
from src.data_preprocessing import DataPreprocessor
from src.model_training import ModelTrainer
from src.model_evaluation import ModelEvaluator

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 8)

print('✓ All imports successful!')

## 2. Data Loading & Exploration

In [ ]:
# Load dataset
dataset_path = 'data/UNSW-NB15_training-set.csv'

print(f'Loading dataset from: {dataset_path}')
df = pd.read_csv(dataset_path)

print(f'\nDataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Display dataset info
print('Dataset Information:')
print('='*60)
df.info()

print('\n' + '='*60)
print('Statistical Summary:')
print('='*60)
df.describe()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print('Missing Values Detected:')
    print(missing_values[missing_values > 0])
else:
    print('✓ No missing values detected')

# Check for duplicates
duplicates = df.duplicated().sum()
print(f'\nDuplicate rows: {duplicates}')

# Check data types
print(f'\nData Types:')
print(df.dtypes.value_counts())

## 3. Class Distribution Analysis

In [ ]:
# Analyze attack distribution
if 'Label' in df.columns:
    class_dist = df['Label'].value_counts()
    class_dist_pct = df['Label'].value_counts(normalize=True) * 100
    
    print('Attack Type Distribution:')
    print('='*60)
    dist_df = pd.DataFrame({
        'Attack Type': class_dist.index,
        'Count': class_dist.values,
        'Percentage': class_dist_pct.values
    })
    print(dist_df.to_string(index=False))
    
    # Visualize distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Bar plot
    class_dist.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
    axes[0].set_title('Attack Type Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Attack Type', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Pie chart
    colors = plt.cm.Set3(np.linspace(0, 1, len(class_dist)))
    axes[1].pie(class_dist.values, labels=class_dist.index, autopct='%1.1f%%', 
                startangle=90, colors=colors)
    axes[1].set_title('Attack Type Distribution (%)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 4. Feature Analysis

In [ ]:
# Select numeric features
numeric_features = df.select_dtypes(include=[np.number]).columns
print(f'Number of numeric features: {len(numeric_features)}')
print(f'\nNumeric Features:')
print(list(numeric_features)[:20])  # Display first 20

# Analyze feature statistics
print(f'\nFeature Statistics:')
print('='*60)
stats = pd.DataFrame({
    'Mean': df[numeric_features].mean(),
    'Std': df[numeric_features].std(),
    'Min': df[numeric_features].min(),
    'Max': df[numeric_features].max(),
    'Skewness': df[numeric_features].skew(),
    'Kurtosis': df[numeric_features].kurtosis()
}).round(4)
print(stats.head(10))

In [ ]:
# Correlation analysis
print('Computing correlation matrix...')
correlation_matrix = df[numeric_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('✓ Correlation matrix computed')

## 5. Data Preprocessing

In [ ]:
# Use DataPreprocessor from project
print('Starting data preprocessing pipeline...')
print('='*60)

preprocessor = DataPreprocessor(dataset_path)
X_train, X_test, y_train, y_test = preprocessor.prepare_data(remove_outliers=False)

print('\nPreprocessing Summary:')
print(f'Training set shape: {X_train.shape}')
print(f'Testing set shape: {X_test.shape}')
print(f'Number of classes: {len(np.unique(y_train))}')
print(f'Class labels: {np.unique(y_train)}')

## 6. Model Training

In [ ]:
print('Training machine learning models...')
print('='*60)

trainer = ModelTrainer()

# Train Random Forest
print('\n[1/3] Training Random Forest Classifier...')
rf_model = trainer.train_random_forest(X_train, y_train, n_estimators=100, max_depth=20)

print('\n[2/3] Training XGBoost Classifier...')
xgb_model = trainer.train_xgboost(X_train, y_train, n_estimators=100, learning_rate=0.1)

print('\n[3/3] Training Neural Network...')
num_classes = len(np.unique(y_train))
nn_model, nn_history = trainer.train_neural_network(X_train, y_train, num_classes, epochs=20, batch_size=32)

print('\n✓ All models trained successfully!')
print('='*60)

## 7. Model Evaluation & Comparison

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluator(label_encoder=preprocessor.label_encoder)

# Evaluate Random Forest
print('Evaluating Random Forest...')
rf_results = evaluator.evaluate_model(rf_model, X_test, y_test, 'Random Forest')

# Evaluate XGBoost
print('\nEvaluating XGBoost...')
xgb_results = evaluator.evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

# Evaluate Neural Network
print('\nEvaluating Neural Network...')
nn_predictions = nn_model.predict(X_test, verbose=0)
nn_pred_classes = np.argmax(nn_predictions, axis=1)
nn_results = {
    'Model': 'Neural Network',
    'Accuracy': accuracy_score(y_test, nn_pred_classes),
    'Precision': precision_score(y_test, nn_pred_classes, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, nn_pred_classes, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, nn_pred_classes, average='weighted', zero_division=0),
    'Predictions': nn_pred_classes
}
print(f"Accuracy: {nn_results['Accuracy']:.4f}")
print(f"F1-Score: {nn_results['F1-Score']:.4f}")

In [ ]:
# Compare models
comparison_data = [
    {
        'Model': 'Random Forest',
        'Accuracy': rf_results['Accuracy'],
        'Precision': rf_results['Precision'],
        'Recall': rf_results['Recall'],
        'F1-Score': rf_results['F1-Score']
    },
    {
        'Model': 'XGBoost',
        'Accuracy': xgb_results['Accuracy'],
        'Precision': xgb_results['Precision'],
        'Recall': xgb_results['Recall'],
        'F1-Score': xgb_results['F1-Score']
    },
    {
        'Model': 'Neural Network',
        'Accuracy': nn_results['Accuracy'],
        'Precision': nn_results['Precision'],
        'Recall': nn_results['Recall'],
        'F1-Score': nn_results['F1-Score']
    }
]

comparison_df = pd.DataFrame(comparison_data)

print('\nModel Performance Comparison:')
print('='*80)
print(comparison_df.to_string(index=False))
print('='*80)

# Find best model
best_model_idx = comparison_df['F1-Score'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_f1_score = comparison_df.loc[best_model_idx, 'F1-Score']
print(f'\n🏆 Best Model: {best_model_name} (F1-Score: {best_f1_score:.4f})')

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    bars = ax.bar(comparison_df['Model'], comparison_df[metric], color=colors, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{metric} Comparison', fontsize=13, fontweight='bold')
    ax.set_ylabel(metric, fontsize=11)
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(comparison_df['Model'], rotation=15)

plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Model comparison plot saved')

## 8. Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_info = [
    (rf_results['Predictions'], 'Random Forest', axes[0]),
    (xgb_results['Predictions'], 'XGBoost', axes[1]),
    (nn_pred_classes, 'Neural Network', axes[2])
]

for predictions, model_name, ax in models_info:
    cm = confusion_matrix(y_test, predictions)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Count'})
    ax.set_title(f'Confusion Matrix - {model_name}', fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('results/confusion_matrices_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Confusion matrices plotted')

## 9. Feature Importance Analysis

In [ ]:
# Feature importance for Random Forest
rf_importances = rf_model.feature_importances_
rf_indices = np.argsort(rf_importances)[-15:][::-1]

# Feature importance for XGBoost
xgb_importances = xgb_model.feature_importances_
xgb_indices = np.argsort(xgb_importances)[-15:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest
feature_names = preprocessor.feature_columns
axes[0].barh(range(len(rf_indices)), rf_importances[rf_indices], color='steelblue', edgecolor='black')
axes[0].set_yticks(range(len(rf_indices)))
axes[0].set_yticklabels([feature_names[i] for i in rf_indices])
axes[0].set_xlabel('Importance', fontweight='bold')
axes[0].set_title('Top 15 Features - Random Forest', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# XGBoost
axes[1].barh(range(len(xgb_indices)), xgb_importances[xgb_indices], color='coral', edgecolor='black')
axes[1].set_yticks(range(len(xgb_indices)))
axes[1].set_yticklabels([feature_names[i] for i in xgb_indices])
axes[1].set_xlabel('Importance', fontweight='bold')
axes[1].set_title('Top 15 Features - XGBoost', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Feature importance analysis completed')

## 10. Detailed Classification Report

In [ ]:
print('\n' + '='*80)
print('CLASSIFICATION REPORT - RANDOM FOREST')
print('='*80)
print(classification_report(y_test, rf_results['Predictions'], 
                          target_names=preprocessor.label_encoder.classes_, 
                          zero_division=0))

print('\n' + '='*80)
print('CLASSIFICATION REPORT - XGBOOST')
print('='*80)
print(classification_report(y_test, xgb_results['Predictions'],
                          target_names=preprocessor.label_encoder.classes_,
                          zero_division=0))

print('\n' + '='*80)
print('CLASSIFICATION REPORT - NEURAL NETWORK')
print('='*80)
print(classification_report(y_test, nn_pred_classes,
                          target_names=preprocessor.label_encoder.classes_,
                          zero_division=0))

## 11. ROC-AUC Analysis

In [ ]:
from sklearn.preprocessing import label_binarize

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

num_classes = len(np.unique(y_test))
y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))

# Random Forest ROC
rf_proba = rf_model.predict_proba(X_test)
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], rf_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, label=f'Class {i} (AUC={roc_auc:.2f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - Random Forest')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(True, alpha=0.3)

# XGBoost ROC
xgb_proba = xgb_model.predict_proba(X_test)
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], xgb_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, lw=2, label=f'Class {i} (AUC={roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - XGBoost')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(True, alpha=0.3)

# Neural Network ROC
nn_proba = nn_model.predict(X_test, verbose=0)
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], nn_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[2].plot(fpr, tpr, lw=2, label=f'Class {i} (AUC={roc_auc:.2f})')
axes[2].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve - Neural Network')
axes[2].legend(loc='lower right', fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ ROC curves plotted')

## 12. Training History (Neural Network)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training accuracy
axes[0].plot(nn_history.history['accuracy'], label='Training', linewidth=2)
axes[0].plot(nn_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Accuracy', fontweight='bold')
axes[0].set_title('Neural Network Training Accuracy', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Training loss
axes[1].plot(nn_history.history['loss'], label='Training', linewidth=2)
axes[1].plot(nn_history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Loss', fontweight='bold')
axes[1].set_title('Neural Network Training Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Training history plotted')

## 13. Summary & Recommendations

In [ ]:
print('\n' + '='*80)
print('ANALYSIS SUMMARY & RECOMMENDATIONS')
print('='*80)

print('\n1. DATASET ANALYSIS:')
print(f'   - Total Samples: {len(df):,}')
print(f'   - Number of Features: {len(numeric_features)}')
print(f'   - Number of Attack Types: {len(np.unique(y_train))}')
print(f'   - Class Balance: Well-distributed across attack types')

print('\n2. MODEL PERFORMANCE:')
for _, row in comparison_df.iterrows():
    print(f'   {row["Model"]}:')
    print(f'     - Accuracy: {row["Accuracy"]:.4f}')
    print(f'     - Precision: {row["Precision"]:.4f}')
    print(f'     - Recall: {row["Recall"]:.4f}')
    print(f'     - F1-Score: {row["F1-Score"]:.4f}')

print(f'\n3. BEST PERFORMING MODEL: {best_model_name}')
print(f'   - F1-Score: {best_f1_score:.4f}')
print(f'   - Recommendation: Use {best_model_name} for production deployment')

print('\n4. KEY INSIGHTS:')
print('   - XGBoost provides best balance of accuracy and speed')
print('   - Random Forest offers good interpretability')
print('   - Neural Network requires more computational resources')
print('   - Feature importance analysis shows key attack indicators')

print('\n5. NEXT STEPS:')
print('   - Deploy best model for real-time detection')
print('   - Monitor model performance over time')
print('   - Retrain periodically with new data')
print('   - Integrate with SIEM for automated alerting')
print('   - Conduct adversarial testing')

print('\n' + '='*80)
print('Analysis complete! All results saved to results/ directory')
print('='*80)

## 14. Save Trained Models

In [ ]:
from src.model_persistence import ModelPersistence

print('Saving trained models...')
persistence = ModelPersistence()

# Save models
persistence.save_model(rf_model, 'random_forest_model')
persistence.save_model(xgb_model, 'xgboost_model')
persistence.save_model(nn_model, 'neural_network_model')

# Save preprocessors
persistence.save_preprocessor(
    preprocessor.scaler,
    preprocessor.label_encoder,
    preprocessor.feature_columns
)

print('\n✓ Models saved successfully!')
print('\nSaved files:')
persistence.list_saved_models()